# 23-24 · Читаем git diff и группируем изменения

Практика к разделу [«Git: от рабочего изменения к коммиту»](../../site/chapters/glava-23/23-28-git-kommit.html).

## Цель

`git diff` не выполняется — его читают. Это упражнение не о запуске кода, а о том, чтобы разобраться, что именно изменилось, и сформулировать по этому изменению короткое, честное commit-сообщение.

## Пример — результат git diff перед коммитом

In [1]:
PRIMER_DIFF = """diff --git a/src/safesort/duplicates.py b/src/safesort/duplicates.py
index 1a2b3c4..5d6e7f8 100644
--- a/src/safesort/duplicates.py
+++ b/src/safesort/duplicates.py
@@ -40,6 +40,9 @@ def find_duplicates(files, chunk_size=DEFAULT_CHUNK_SIZE):
     for size, candidates in by_size.items():
         if len(candidates) < 2:
             continue
+
+        if size == 0:
+            logger.info("Пустые файлы тоже считаются дубликатами: %d штук", len(candidates))
         by_digest = defaultdict(list)
diff --git a/tests/test_duplicates.py b/tests/test_duplicates.py
index 9f8e7d6..2c3b4a5 100644
--- a/tests/test_duplicates.py
+++ b/tests/test_duplicates.py
@@ -12,3 +12,10 @@ def test_identical_content_files_are_grouped(tmp_path):
     assert len(groups) == 1
     assert len(groups[0].files) == 2
+
+
+def test_empty_files_are_duplicates_of_each_other(tmp_path):
+    (tmp_path / "a.txt").write_text("")
+    (tmp_path / "b.txt").write_text("")
+    groups = find_duplicates(scan(tmp_path, Config()))
+    assert len(groups) == 1
"""

print(PRIMER_DIFF)

diff --git a/src/safesort/duplicates.py b/src/safesort/duplicates.py
index 1a2b3c4..5d6e7f8 100644
--- a/src/safesort/duplicates.py
+++ b/src/safesort/duplicates.py
@@ -40,6 +40,9 @@ def find_duplicates(files, chunk_size=DEFAULT_CHUNK_SIZE):
     for size, candidates in by_size.items():
         if len(candidates) < 2:
             continue
+
+        if size == 0:
+            logger.info("Пустые файлы тоже считаются дубликатами: %d штук", len(candidates))
         by_digest = defaultdict(list)
diff --git a/tests/test_duplicates.py b/tests/test_duplicates.py
index 9f8e7d6..2c3b4a5 100644
--- a/tests/test_duplicates.py
+++ b/tests/test_duplicates.py
@@ -12,3 +12,10 @@ def test_identical_content_files_are_grouped(tmp_path):
     assert len(groups) == 1
     assert len(groups[0].files) == 2
+
+
+def test_empty_files_are_duplicates_of_each_other(tmp_path):
+    (tmp_path / "a.txt").write_text("")
+    (tmp_path / "b.txt").write_text("")
+    groups = find_duplicates(scan(tmp_path, Config(

## Разбираем diff построчно

In [2]:
stroki = PRIMER_DIFF.splitlines()

izmenennye_fajly = [s.split()[-1][2:] for s in stroki if s.startswith("diff --git")]
dobavlennye_stroki = [s for s in stroki if s.startswith("+") and not s.startswith("+++")]
udalennye_stroki = [s for s in stroki if s.startswith("-") and not s.startswith("---")]

print("Изменённые файлы:", izmenennye_fajly)
print("Добавлено строк:", len(dobavlennye_stroki))
print("Удалено строк:", len(udalennye_stroki))

Изменённые файлы: ['src/safesort/duplicates.py', 'tests/test_duplicates.py']
Добавлено строк: 10
Удалено строк: 0


## Проверка

In [3]:
assert izmenennye_fajly == ["src/safesort/duplicates.py", "tests/test_duplicates.py"]
assert len(dobavlennye_stroki) > 0
assert len(udalennye_stroki) == 0  # в этом diff ничего не удалено, только добавлено
print("Верно: diff затронул два файла, и в нём только добавления.")

Верно: diff затронул два файла, и в нём только добавления.


## Задание ★ Базовая практика

Прочитайте diff выше и сформулируйте commit-сообщение, которое описывало бы это изменение одной строкой в духе Conventional Commits (например, `feat: ...` или `test: ...`). Запишите его в переменную `moe_commit_soobshenie` и проверьте, что оно не пустое и начинается с одного из принятых префиксов.

In [4]:
moe_commit_soobshenie = "feat: treat zero-byte files as duplicates of each other"

dopustimye_prefiksy = ("feat:", "fix:", "test:", "docs:", "refactor:", "chore:")

assert moe_commit_soobshenie.strip() != ""
assert moe_commit_soobshenie.startswith(dopustimye_prefiksy)
print("Верно:", moe_commit_soobshenie)

Верно: feat: treat zero-byte files as duplicates of each other
